In [2]:
!pip install tenacity google-genai -q

In [3]:
!git clone https://github.com/sl2902/sdrf.git

Cloning into 'sdrf'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (158/158), done. (97/158)
remote: Compressing objects: 100% (108/108), done.
remote: Total 350 (delta 86), reused 120 (delta 49), pack-reused 192 (from 1)
Receiving objects: 100% (350/350), 105.53 KiB | 1.43 MiB/s, done.
Resolving deltas: 100% (192/192), done.


In [4]:
import asyncio, logging
import json
import os
import pandas as pd

from sdrf.prompts import v1 as prompt
from sdrf.prompts import v2 as prompt2
from sdrf.prompts import v3 as prompt3
from sdrf.models.openai_client import OpenAIClient
from sdrf.models.gemini_client import GeminiClient
from sdrf.pipeline import run_extraction
from sdrf.submission import merge_results, save_submission

from kaggle_secrets import UserSecretsClient
from google.oauth2 import service_account

In [6]:
def get_credentials(secret_name: str = "gcp_service_account") -> service_account.Credentials:
    """Fetch GCP Credentials"""
    user_secrets =  UserSecretsClient()
    service_account_json = user_secrets.get_secret(secret_name)

    with open("/tmp/service_account.json", "w") as f:
        f.write(service_account_json)
    credentials_info = json.loads(service_account_json)
    credentials = service_account.Credentials.from_service_account_info(
        credentials_info,
        scopes=[
            "https://www.googleapis.com/auth/cloud-platform",
            "https://www.googleapis.com/auth/generative-language",
        ]
    )



    os.environ["project_id"] = credentials_info["project_id"]
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/tmp/service_account.json"


    return credentials_info, credentials

user_secrets, credentials = get_credentials()
secrets = UserSecretsClient()

In [ ]:
TWO_PASS      = False   # set True to enable verification pass
MAX_CONCURRENT = 1     # reduce if hitting rate limits

In [ ]:
model = OpenAIClient(
    api_key=secrets.get_secret("OPENAI_API_KEY"),
    model="gpt-4o",
)

In [ ]:
gpt4o_results = await run_extraction(
    model=model,
    prompt=prompt3,
    two_pass=TWO_PASS,
    max_concurrent=MAX_CONCURRENT,
    model_name="gpt_4o",
)

In [ ]:
model = GeminiClient(
    project=user_secrets["project_id"],
    credentials=credentials,
    model="gemini-2.5-pro",
)

In [ ]:
gemini_results = await run_extraction(
    model=model,
    prompt=prompt3,
    two_pass=TWO_PASS,
    max_concurrent=MAX_CONCURRENT,
    model_name="gemini_pro",
)

In [ ]:
if gemini_results and gpt4o_results:
    merged = merge_results(gemini_results, gpt4o_results)
elif gemini_results:
    merged = gemini_results
elif gpt4o_results:
    merged = gpt4o_results
save_submission(merged, two_pass=TWO_PASS)